# Creating a Baseline Model Using Simple Heuristics

A baseline model is a simple reference point that tells you whether a real machine learning model is actually adding value. It is intentionally naive, but it must be measured correctly and on held-out data.

This lesson covers simple classification and regression baselines, how to evaluate them, and how to avoid leakage while doing so.

## Why baselines matter

Without a baseline, a model score has no context. A baseline helps you determine whether your model is better than a trivial solution, whether class imbalance is hiding weak performance, and whether your features contain meaningful signal.

Common baseline strategies include:

- Majority class prediction
- Stratified random prediction
- Uniform random prediction
- Mean or median prediction for regression
- Simple rule-based heuristics

In [ ]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(31)
n_rows = 500

classification_df = pd.DataFrame({
    'tenure': rng.integers(1, 72, size=n_rows),
    'MonthlyCharges': rng.normal(70, 15, size=n_rows).round(2),
    'TotalCharges': rng.normal(2500, 800, size=n_rows).round(2),
})
classification_logit = -0.03 * classification_df['tenure'] + 0.015 * classification_df['MonthlyCharges']
classification_prob = 1 / (1 + np.exp(-classification_logit))
classification_df['Churn'] = (rng.random(n_rows) < classification_prob).astype(int)

regression_df = pd.DataFrame({
    'size_sqft': rng.integers(500, 3500, size=n_rows),
    'rooms': rng.integers(1, 6, size=n_rows),
    'age_years': rng.integers(1, 40, size=n_rows),
})
regression_df['price'] = (regression_df['size_sqft'] * 1200 + regression_df['rooms'] * 10000 - regression_df['age_years'] * 1500 + rng.normal(0, 20000, size=n_rows)).round(2)

classification_df.head(), regression_df.head()

## Classification baseline

A majority-class baseline predicts the most common class in the training data for every row. It is a useful floor for classification problems, especially when classes are imbalanced.

A stratified random baseline and a uniform random baseline are also useful sanity checks.

In [ ]:
TARGET_COLUMN = 'Churn'
X = classification_df.drop(columns=[TARGET_COLUMN])
y = classification_df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_preds = baseline.predict(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

results = {
    'baseline_accuracy': accuracy_score(y_test, baseline_preds),
    'baseline_f1': f1_score(y_test, baseline_preds, average='weighted', zero_division=0),
    'model_accuracy': accuracy_score(y_test, model_preds),
    'model_f1': f1_score(y_test, model_preds, average='weighted'),
}

print({k: round(v, 3) for k, v in results.items()})
print('\n--- Baseline Report ---')
print(classification_report(y_test, baseline_preds, zero_division=0))
print('--- Model Report ---')
print(classification_report(y_test, model_preds, zero_division=0))

## Regression baseline

For regression, the standard baseline is to predict the training-set mean or median for every row. This gives you a clear reference point for MSE, MAE, and R².

If a real regression model cannot beat this baseline, the feature set or modeling approach needs review.

In [ ]:
TARGET_COLUMN = 'price'
X_reg = regression_df.drop(columns=[TARGET_COLUMN])
y_reg = regression_df[TARGET_COLUMN]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg_baseline = DummyRegressor(strategy='mean')
reg_baseline.fit(X_train_reg, y_train_reg)
baseline_reg_preds = reg_baseline.predict(X_test_reg)

reg_model = LinearRegression()
reg_model.fit(X_train_reg, y_train_reg)
model_reg_preds = reg_model.predict(X_test_reg)

reg_results = {
    'baseline_mse': mean_squared_error(y_test_reg, baseline_reg_preds),
    'baseline_mae': mean_absolute_error(y_test_reg, baseline_reg_preds),
    'baseline_r2': r2_score(y_test_reg, baseline_reg_preds),
    'model_mse': mean_squared_error(y_test_reg, model_reg_preds),
    'model_mae': mean_absolute_error(y_test_reg, model_reg_preds),
    'model_r2': r2_score(y_test_reg, model_reg_preds),
}

print({k: round(v, 3) for k, v in reg_results.items()})

## Practical rules

- Split before fitting any baseline.
- Use DummyClassifier or DummyRegressor for honest comparisons.
- Compare the model and baseline on the same metric.
- Inspect per-class metrics, not just accuracy.
- Treat a strong baseline as a signal to investigate the data, not as a reason to skip modeling discipline.

## Bonus resources

- Scikit-learn Dummy Models Documentation
- Scikit-learn Model Evaluation Metrics Guide